## Data Transformation Process

In [ ]:
import os
# os.chdir('../')
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\TextSummarizer'

## Data Configuartion
`src/textsummarizer/entity/config_entity.py`

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

`src/textsummarizer/config/configuration.py`

In [3]:
from src.textsummarizer.constants import *
from src.textsummarizer.utils.main import read_yaml, create_directories

[ 2026-05-13 20:44:16,802 : INFO : __init__ : Logger initialized successfully ]


In [4]:
class ConfigurationManager:
    def __init__(self, 
                 config_path=CONFIG_FILE_PATH,
                 params_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir= config.root_dir,
            data_path= config.data_path,
            tokenizer_name= config.tokenizer_name
        )

## Components
`src/textsummarizer/components/datatransformation.py`

In [ ]:
from src.textsummarizer.logging import logging

from transformers import AutoTokenizer
from datasets import load_from_disk

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
    
    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length=1024,
            truncation=True
        )

        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True
        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))

In [6]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config= data_transformation_config)
data_transformation.convert()

[ 2026-05-13 20:55:10,615 : INFO : main : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-13 20:55:10,622 : INFO : main : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-13 20:55:10,625 : INFO : main : Create Directory at: artifacts ]
[ 2026-05-13 20:55:10,628 : INFO : main : Create Directory at: artifacts/data_transformation ]
[ 2026-05-13 20:55:11,302 : INFO : _client : HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect" ]


[ 2026-05-13 20:55:11,305 : WARNING : _http : Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads. ]
[ 2026-05-13 20:55:11,335 : INFO : _client : HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK" ]
[ 2026-05-13 20:55:11,376 : INFO : _client : HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK" ]


c:\Users\singh\OneDrive\Desktop\TextSummarizer\TSenv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\singh\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[ 2026-05-13 20:55:11,751 : INFO : _client : HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect" ]
[ 2026-05-13 20:55:11,779 : INFO : _client : HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK" ]
[ 2026-05-13 20:55:11,811 : INFO : _client : HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK" ]
[ 2026-05-13 20:55:12,110 : INFO : _client : HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found" ]
[ 2026-05-13 20:55:12,378 : INFO : _client : HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=tr

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 76547.62 examples/s]
